In [24]:
!nvidia-smi

Sun Sep 20 06:54:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 227 not upgraded.


In [26]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [27]:
!ollama --version

]11;?\Warning: could not connect to a running Ollama instance


In [28]:
import subprocess
import time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)
print("Ollama started")

Ollama started


In [29]:
import requests

r = requests.get("http://127.0.0.1:11434")
print(r.status_code)
print(r.text)

200
Ollama is running


In [30]:
!ollama pull qwen3:8b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling a3de86cd1c13: 100% ▕██████████████████▏ 5.2 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 05a61d37b084: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [31]:
!ollama list

]11;?\NAME        ID              SIZE      MODIFIED      
qwen3:8b    500a1f067a9f    5.2 GB    5 seconds ago    


In [32]:
import requests

r = requests.get("http://127.0.0.1:11434/v1/models")

print("Status:", r.status_code)
print(r.text)

Status: 200
{"object":"list","data":[{"id":"qwen3:8b","object":"model","created":1789887298,"owned_by":"library"}]}



In [33]:
import requests

r = requests.post(
    "http://127.0.0.1:11434/v1/chat/completions",
    json={
        "model": "qwen3:8b",
        "messages": [
            {
                "role": "user",
                "content": "Say hello from the Kaggle GPU in one sentence."
            }
        ]
    },
    timeout=120
)

print("Status:", r.status_code)
print(r.text)

Status: 200
{"id":"chatcmpl-94","object":"chat.completion","created":1789887316,"model":"qwen3:8b","system_fingerprint":"fp_ollama","choices":[{"index":0,"message":{"role":"assistant","content":"Hello from the Kaggle GPU! I'm here to help you tackle your machine learning challenges!","reasoning":"Okay, the user wants me to say hello from the Kaggle GPU in one sentence. Let me start by understanding the context. Kaggle is a platform for data science competitions, and they offer GPUs for running notebooks. So the user probably wants a greeting that mentions the GPU environment.\n\nFirst, I need to make sure the sentence is concise. Maybe start with a standard greeting like \"Hello!\" Then mention the Kaggle GPU. Should I specify the type of GPU or just say \"Kaggle GPU\"? Since the user said \"in one sentence,\" keeping it simple is better. \n\nAlso, maybe add a friendly note about being ready to work. Something like \"I'm here to help you tackle your machine learning challenges!\" That 

In [34]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 125211 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.1) over (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [35]:
!cloudflared --version

cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)


In [36]:
import subprocess
import time

cloudflare_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:11434",
        "--http-host-header", "localhost:11434"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Cloudflare tunnel started")

# Read initial output
for _ in range(10):
    line = cloudflare_process.stdout.readline()
    if line:
        print(line.strip())

Cloudflare tunnel started
2026-09-20T06:55:19Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-20T06:55:19Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-20T06:55:23Z INF +--------------------------------------------------------------------------------------------+
2026-09-20T06:55:23Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-20T06:55:23Z INF |  https://boards-colony-women

In [37]:
import requests

try:
    r = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
    print("Status:", r.status_code)
    print(r.text)
except Exception as e:
    print("Error:", e)

Status: 200
{"models":[{"name":"qwen3:8b","model":"qwen3:8b","modified_at":"2026-09-20T06:54:58.932688092Z","size":5225388164,"digest":"500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"8.2B","quantization_level":"Q4_K_M","context_length":40960,"embedding_length":4096},"capabilities":["completion","tools","thinking"]}]}


In [38]:
import requests

PUBLIC_URL = "https://reunion-detailed-maintains-cool.trycloudflare.com"

r = requests.get(PUBLIC_URL + "/v1/models")

print("Status:", r.status_code)
print(r.text)

Status: 200
{"object":"list","data":[{"id":"qwen3:8b","object":"model","created":1789887298,"owned_by":"library"}]}



In [40]:
import requests

PUBLIC_URL = "https://reunion-detailed-maintains-cool.trycloudflare.com"

r = requests.post(
    PUBLIC_URL + "/v1/chat/completions",
    json={
        "model": "qwen3:8b",
        "messages": [
            {
                "role": "user",
                "content": "Hello. Confirm that you are running through the Cloudflare tunnel."
            }
        ]
    },
    timeout=120
)

print("Status:", r.status_code)
print(r.text)

Status: 200
{"id":"chatcmpl-756","object":"chat.completion","created":1789887394,"model":"qwen3:8b","system_fingerprint":"fp_ollama","choices":[{"index":0,"message":{"role":"assistant","content":"Hello! I am a large language model and do not run through Cloudflare tunnels or any other network infrastructure. My interactions occur through an API that may or may not use Cloudflare for traffic management or security, but the model itself does not operate within a tunnel. Let me know if you have other questions!","reasoning":"Okay, the user is asking me to confirm that I'm running through a Cloudflare tunnel. Let me think about how to approach this.\n\nFirst, I need to recall what Cloudflare Tunnel does. From what I know, Cloudflare Tunnel is a service that allows you to securely expose internal services to the internet without opening firewall ports. It uses a secure tunnel to connect to Cloudflare's DNS and then routes traffic to your internal services. So, if a service is running throug